# 01 — Download Twin-2K-500 and lock the data splits

This notebook performs **steps 2 and 3** explicitly, cell by cell: download the official source files, assign participants, define question roles, and save separate data partitions. The decisions and splitting code live here rather than inside a hidden pipeline.

**Population:** the 2,058 US Prolific participants who completed all four waves in early 2025. This is a cohort, not a guaranteed probability sample of current US opinion.

**Design:** 1,658 shared-training participants, 300 development participants and 100 final personas. Each development/final persona may use up to 92 specified earlier-wave answers. The shared model may use 430 eligible earlier-wave response columns for background respondents.

**Held-out families:** paid leave, wealth tax and maximization for validation; three climate-policy questions for final testing. Their earlier repetitions are also forbidden inputs. All perceived-public-support sliders (`QID290`) are excluded.

Held-out labels must be read programmatically to partition them, but are **never displayed, summarized by response category, or used to choose the split**. Raw downloads still contain every answer; downstream training should load only the explicitly named processed files. Folder separation is an experimental convention, not an access-control mechanism.

Sources: [dataset](https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500), [paper](https://arxiv.org/abs/2505.17479), [catalogue documentation](https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500/blob/main/question_catalog_and_human_response_csv/question_catalog_README.md). Dataset licence: **CC BY 4.0**; retain attribution when sharing derived work.


## 1. Project paths and small file helpers

Run this notebook with the project's `.venv` kernel. Reruns reuse the pinned download and must reproduce existing manifests and partitions; conflicting outputs cause an error rather than silently replacing a split.


In [1]:
import hashlib
import json
import shutil
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download
# Works when launched from the project folder or from notebooks/.
ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "pyproject.toml").is_file()
             and (path / "notebooks").is_dir()), None)
assert ROOT is not None, "Open this notebook inside AS_Tech_Task."
SEED = 20261023

for name, relative in {
    "HF_HOME": ".cache/huggingface",
    "MPLCONFIGDIR": ".cache/matplotlib",
    "XDG_CACHE_HOME": ".cache",
    "JUPYTER_RUNTIME_DIR": ".cache/jupyter/runtime",
}.items():
    os.environ.setdefault(name, str(ROOT / relative))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

RAW = ROOT / "data/raw/twin_2k_500"
SPLITS = ROOT / "data/splits"
PROCESSED = ROOT / "data/processed"
HELDOUT = ROOT / "data/heldout"
REPORTS = ROOT / "outputs"
for path in [RAW, SPLITS, PROCESSED, HELDOUT, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_fixed_json(path, payload):
    path = Path(path)
    if path.exists():
        assert json.loads(path.read_text()) == payload, f"Existing manifest differs: {path.name}"
    else:
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def save_fixed_csv(path, frame):
    content = frame.to_csv(index=False)
    if path.exists():
        assert path.read_text() == content, f"Existing CSV differs: {path.name}"
    else:
        path.write_text(content)

def save_fixed_parquet(path, frame):
    if path.exists():
        pd.testing.assert_frame_equal(pd.read_parquet(path), frame)
    else:
        frame.to_parquet(path, index=False)

print(f"Project: {ROOT.name}; fixed seed: {SEED}")


Project: AS_Tech_Task; fixed seed: 20261023


/Users/louismcauliffe/code/projects/AS_Tech_Task/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Download the official structured dataset at one immutable revision

We need the complete numeric and labelled response tables for all participants and waves, plus the catalogue and documentation. We do not download supplied persona summaries or precomputed LLM simulations. The Hugging Face commit, original URLs, file sizes and SHA-256 checksums are recorded. Original files are stored read-only.

The first run resolves `main` to a commit. Later runs verify the existing copies without fetching a newer version.


In [2]:
REPO_ID = "LLM-Digital-Twin/Twin-2K-500"
TABLE_DIRECTORY = "question_catalog_and_human_response_csv"
REQUIRED_FILES = [
    "README.md",
    f"{TABLE_DIRECTORY}/question_catalog_README.md",
    f"{TABLE_DIRECTORY}/question_catalog.json",
    f"{TABLE_DIRECTORY}/wave1_3_response.csv",
    f"{TABLE_DIRECTORY}/wave1_3_response_label.csv",
    f"{TABLE_DIRECTORY}/wave4_response.csv",
    f"{TABLE_DIRECTORY}/wave4_response_label.csv",
]
SOURCE_MANIFEST = RAW / "source_manifest.json"

if SOURCE_MANIFEST.exists():
    source_manifest = json.loads(SOURCE_MANIFEST.read_text())
    assert source_manifest["repo_id"] == REPO_ID
    REVISION = source_manifest["revision"]
    assert {entry["repo_path"] for entry in source_manifest["files"]} == set(REQUIRED_FILES)
    for entry in source_manifest["files"]:
        local_path = ROOT / entry["local_path"]
        assert sha256_file(local_path) == entry["sha256"], f"Source checksum mismatch: {entry['repo_path']}"
else:
    info = HfApi(token=False).dataset_info(REPO_ID)
    REVISION = info.sha
    assert REVISION and len(REVISION) == 40
    available = {entry.rfilename for entry in info.siblings}
    assert set(REQUIRED_FILES).issubset(available)
    entries = []
    for repo_path in REQUIRED_FILES:
        cached_path = Path(hf_hub_download(
            repo_id=REPO_ID, repo_type="dataset", filename=repo_path,
            revision=REVISION, token=False, cache_dir=ROOT / ".cache/huggingface/hub",
        ))
        target = RAW / REVISION / repo_path
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert sha256_file(target) == sha256_file(cached_path)
        else:
            shutil.copyfile(cached_path, target)
        target.chmod(0o444)
        entries.append({
            "repo_path": repo_path,
            "local_path": str(target.relative_to(ROOT)),
            "url": f"https://huggingface.co/datasets/{REPO_ID}/resolve/{REVISION}/{repo_path}",
            "bytes": target.stat().st_size,
            "sha256": sha256_file(target),
        })
    source_manifest = {
        "repo_id": REPO_ID, "revision": REVISION, "licence": "CC-BY-4.0",
        "attribution": "Toubia et al. (2025), Twin-2K-500, arXiv:2505.17479",
        "files": entries,
    }
    save_fixed_json(SOURCE_MANIFEST, source_manifest)

SNAPSHOT = RAW / REVISION
TABLES = SNAPSHOT / TABLE_DIRECTORY
save_fixed_json(ROOT / "configs/data_source.json", source_manifest)
display(pd.DataFrame([
    {"file": entry["repo_path"].split("/")[-1], "bytes": entry["bytes"], "sha256": entry["sha256"]}
    for entry in source_manifest["files"]
]))
print(f"Pinned Hugging Face commit: {REVISION}")


,file,bytes,sha256
0,README.md,10970,9f71a028d2510476cdb3dd365f4525e7b7cc0736f17939...
1,question_catalog_README.md,10078,eaf3ccb2a5a0382723a057d8e05a0d3a9464c8c9cef1a6...
2,question_catalog.json,214137,e72e305f0c7c83bb990e97de4c0cf2c718ead18cc19723...
3,wave1_3_response.csv,5462260,fc227b1682e2654fd4b2a8152294b68118c0e3be4d83f4...
4,wave1_3_response_label.csv,21102001,4d735be9e5f00d072d17be25029cef524de9ec383911a4...
5,wave4_response.csv,633807,09ec1a88bdd48fec95963da3102df0ca467378bf729cbc...
6,wave4_response_label.csv,4728435,02eb7eeff000406ff70c954565dada7a4a8d997c2f8152...


Pinned Hugging Face commit: f883165a3026fde855dfd448e0cd16443ab257b6


## 3. Inspect schema and participant IDs only

At this point we read column headers, question metadata and IDs, not response values. Answer-level CSV column names are unique; raw QuestionID strings are not. Matrix row suffixes are not always consecutive.


In [3]:
catalogue = json.loads((TABLES / "question_catalog.json").read_text())
wave13_path = TABLES / "wave1_3_response.csv"
wave4_path = TABLES / "wave4_response.csv"
wave13_columns = pd.read_csv(wave13_path, nrows=0).columns.tolist()
wave4_columns = pd.read_csv(wave4_path, nrows=0).columns.tolist()
wave13_ids = pd.read_csv(wave13_path, usecols=["pid"], dtype={"pid": "string"})["pid"]
wave4_ids = pd.read_csv(wave4_path, usecols=["pid"], dtype={"pid": "string"})["pid"]
assert wave13_ids.notna().all() and wave4_ids.notna().all()
assert wave13_ids.is_unique and wave4_ids.is_unique
assert set(wave13_ids) == set(wave4_ids)
assert len(wave13_ids) == 2058

column_to_question = {}
for question in catalogue:
    for column in question.get("csv_columns", []):
        assert column not in column_to_question, f"Duplicate answer column: {column}"
        column_to_question[column] = question
assert set(wave13_columns) - {"pid"} == set(column_to_question)
assert set(wave4_columns).issubset(set(wave13_columns))
display(pd.DataFrame([
    {"source": "waves 1–3", "participants": len(wave13_ids), "answer columns": len(wave13_columns)-1},
    {"source": "wave 4", "participants": len(wave4_ids), "answer columns": len(wave4_columns)-1},
]))
print(f"Catalogue entries: {len(catalogue)}; distinct raw QID strings: {len({q['QuestionID'] for q in catalogue})}")


,source,participants,answer columns
0,waves 1–3,2058,760
1,wave 4,2058,126


Catalogue entries: 256; distinct raw QID strings: 246


## 4. Assign respondents explicitly

Sort IDs as strings, shuffle them once using NumPy's seeded generator, then take consecutive slices. No selection or stratification uses answers. Each final persona has equal weight; this does not create national survey weights.


In [4]:
all_ids = np.array(sorted(wave13_ids.tolist()), dtype=str)
shuffled_ids = np.random.default_rng(SEED).permutation(all_ids)
train_ids = shuffled_ids[:1658].tolist()
development_ids = shuffled_ids[1658:1958].tolist()
final_ids = shuffled_ids[1958:].tolist()

respondent_groups = {
    "shared_training": train_ids,
    "development": development_ids,
    "final_personas": final_ids,
}
assert [len(ids) for ids in respondent_groups.values()] == [1658, 300, 100]
assert len(set(train_ids) | set(development_ids) | set(final_ids)) == 2058
assert not set(train_ids) & set(development_ids)
assert not set(train_ids) & set(final_ids)
assert not set(development_ids) & set(final_ids)

respondent_manifest = {
    "seed": SEED, "source_revision": REVISION,
    "algorithm": "Lexicographically sort pid strings; numpy default_rng(seed).permutation; slices [0:1658], [1658:1958], [1958:2058].",
    "numpy_version": np.__version__, "groups": respondent_groups,
}
assignments = pd.DataFrame([
    {"pid": pid, "split": name, "position_in_split": position}
    for name, ids in respondent_groups.items()
    for position, pid in enumerate(ids)
])
save_fixed_json(SPLITS / "respondent_split.json", respondent_manifest)
save_fixed_csv(SPLITS / "respondent_assignments.csv", assignments)
display(assignments.groupby("split", sort=False).size().rename("participants").to_frame())


,participants
split,
shared_training,1658
development,300
final_personas,100


## 5. Declare the held-out question families

**Validation:** parental paid leave and wealth tax (wave-4 answers), plus the six-item maximization scale (earlier-wave answers). **Final test:** the three climate-policy questions (wave-4 answers).

These families are withheld from shared-model training for **all** respondents. Earlier answers to the repeated policy questions are also forbidden. `QID290` estimates of public support are excluded entirely. These are exact known repetition/proxy exclusions; a full semantic audit and task-context reconstruction remain part of the next data-preparation step.


In [5]:
VALIDATION_POLICY = ["QID287_7", "QID287_10"]
VALIDATION_MAXIMIZATION = [f"QID239_{i}" for i in range(1, 7)]
FINAL_TEST = ["QID287_1", "QID287_2", "QID287_3"]
PERMITTED_POLICY_HISTORY = ["QID287_4", "QID287_5", "QID287_6", "QID287_11", "QID287_12"]
PUBLIC_SUPPORT_PROXIES = [column for column in column_to_question if column.startswith("QID290_")]
FORBIDDEN_INPUT_COLUMNS = set(VALIDATION_POLICY + VALIDATION_MAXIMIZATION + FINAL_TEST + PUBLIC_SUPPORT_PROXIES)

assert len(PUBLIC_SUPPORT_PROXIES) == 10
assert set(VALIDATION_POLICY + FINAL_TEST).issubset(wave4_columns)
assert set(VALIDATION_MAXIMIZATION).issubset(wave13_columns)
assert not set(VALIDATION_MAXIMIZATION) & set(wave4_columns)
display(pd.DataFrame([
    {"role": "validation policy", "question_ids": ", ".join(VALIDATION_POLICY), "label_source": "wave 4"},
    {"role": "validation maximization", "question_ids": "QID239_1 to QID239_6", "label_source": "waves 1–3"},
    {"role": "final climate test", "question_ids": ", ".join(FINAL_TEST), "label_source": "wave 4"},
]))


,role,question_ids,label_source
0,validation policy,"QID287_7, QID287_10",wave 4
1,validation maximization,QID239_1 to QID239_6,waves 1–3
2,final climate test,"QID287_1, QID287_2, QID287_3",wave 4


## 6. Declare the 92 persona-evidence columns

The same evidence bundle is used for development and final personas, and later for the prompted comparison. Missing answers remain missing. The five permitted healthcare/immigration answers provide additional policy context without revealing the validation or climate-test opinions.


In [6]:
anchor_groups = {
    "demographics": [f"QID{i}" for i in range(11, 25)],
    "agentic_communal_values": [f"QID29_{i}" for i in range(1, 25)],
    "individualism_collectivism": [f"QID233_{i}" for i in range(1, 17)],
    "environmental_values": [f"QID235_{i}" for i in range(1, 7)],
    "spending_tendencies": [f"QID{i}" for i in range(31, 35)],
    "delayed_payment_choices": [f"QID84_{i}" for i in range(4, 13)],
    "lottery_choices": [f"QID250_{i}" for i in range(1, 15)],
    "permitted_policy_history": PERMITTED_POLICY_HISTORY,
}
ANCHORS = [column for columns in anchor_groups.values() for column in columns]
assert len(ANCHORS) == len(set(ANCHORS)) == 92
assert set(ANCHORS).issubset(wave13_columns)
assert not set(ANCHORS) & FORBIDDEN_INPUT_COLUMNS
display(pd.DataFrame([
    {"evidence": name, "columns": len(columns)}
    for name, columns in anchor_groups.items()
]))
print(f"Total permitted persona-evidence columns: {len(ANCHORS)}")


,evidence,columns
0,demographics,14
1,agentic_communal_values,24
2,individualism_collectivism,16
3,environmental_values,6
4,spending_tendencies,4
5,delayed_payment_choices,9
6,lottery_choices,14
7,permitted_policy_history,5


Total permitted persona-evidence columns: 92


## 7. Build the shared-training question whitelist from metadata

Include single-select demographics, nonclinical personality/values, economic preferences, the five permitted policy items, and eight risk/benefit ratings. Exclude objective cognitive tests, image/pricing tasks, free text, sliders, clinical instruments and unsupported response formats.

This reproduces the proposed **430-column** whitelist using readable rules. It is an answer-column selection, not yet a cleaned question bank. Long batteries and price lists will need block-balanced weights during training.


In [7]:
TRAINING_COLUMNS = []
training_counts = {}
for question in catalogue:
    block = question["BlockName"].strip()
    qid = question["QuestionID"]
    kind = question["QuestionType"]
    settings = question.get("Settings", {})
    selector = settings.get("Selector")
    permitted_block = (
        block in {"Demographics", "Personality", "Economic preferences"}
        or qid in {"QID287", "QID288", "QID289"}
    )
    single_select = (
        (kind == "MC" and selector in {"SAVR", "SAHR"})
        or (kind == "Matrix" and (
            selector == "Bipolar"
            or (selector == "Likert" and settings.get("SubSelector") == "SingleAnswer")
        ))
    )
    # QID125 is an anxiety symptom battery; QID148 belongs to the clinical inventory.
    # Other clinical multi-select items are excluded by single_select above.
    if permitted_block and single_select and qid not in {"QID125", "QID148"}:
        permitted = [c for c in question["csv_columns"] if c not in FORBIDDEN_INPUT_COLUMNS]
        TRAINING_COLUMNS.extend(permitted)
        training_counts[block] = training_counts.get(block, 0) + len(permitted)

assert len(TRAINING_COLUMNS) == len(set(TRAINING_COLUMNS)) == 430
assert set(TRAINING_COLUMNS).issubset(wave13_columns)
assert set(ANCHORS).issubset(TRAINING_COLUMNS)
assert not set(TRAINING_COLUMNS) & FORBIDDEN_INPUT_COLUMNS
display(pd.DataFrame(list(training_counts.items()), columns=["domain", "permitted answer columns"]))


,domain,permitted answer columns
0,Demographics,14
1,Personality,227
2,Economic preferences,176
3,False consensus,5
4,Non-experimental heuristics and biases,8


## 8. Save a question-role manifest and the agreed split configuration

Every source answer column receives a role. Persona-evidence columns are a subset of training questions, but development/final people contribute no answers to shared-model fitting. Identifiers are saved before reading answer values.


In [8]:
question_rows = []
for column, question in column_to_question.items():
    if column in FINAL_TEST:
        role = "final_test_only"
    elif column in VALIDATION_POLICY:
        role = "validation_policy_only"
    elif column in VALIDATION_MAXIMIZATION:
        role = "validation_maximization_only"
    elif column in PUBLIC_SUPPORT_PROXIES:
        role = "excluded_public_support_proxy"
    elif column in TRAINING_COLUMNS:
        role = "shared_training"
    else:
        role = "excluded_other"
    question_rows.append({
        "answer_column": column, "question_id": question["QuestionID"],
        "block_name": question["BlockName"].strip(), "question_type": question["QuestionType"],
        "role": role, "persona_evidence": column in ANCHORS,
        "present_wave1_3": column in wave13_columns, "present_wave4": column in wave4_columns,
    })
question_assignments = pd.DataFrame(question_rows)
question_manifest = {
    "source_revision": REVISION, "training_wave": "wave1_3",
    "training_columns": TRAINING_COLUMNS, "anchor_groups": anchor_groups,
    "validation_policy": {"columns": VALIDATION_POLICY, "label_wave": "wave4"},
    "validation_maximization": {"columns": VALIDATION_MAXIMIZATION, "label_wave": "wave1_3"},
    "final_test": {"columns": FINAL_TEST, "label_wave": "wave4"},
    "forbidden_input_columns_all_respondents_all_waves": sorted(FORBIDDEN_INPUT_COLUMNS),
    "exclude_supplied_persona_summaries": True,
    "heldout_policy_repetitions": "Earlier-wave answers are excluded too, even for background training people.",
}
save_fixed_json(SPLITS / "question_split.json", question_manifest)
save_fixed_csv(SPLITS / "question_assignments.csv", question_assignments)
save_fixed_json(ROOT / "configs/data_split_plan.json", {
    "seed": SEED, "source_revision": REVISION,
    "respondent_counts": {name: len(ids) for name, ids in respondent_groups.items()},
    "respondent_ordering": respondent_manifest["algorithm"],
    "question_split": question_manifest,
})
display(question_assignments.groupby("role").size().rename("columns").to_frame())
print("Participant and question manifests saved before reading response values.")


,columns
role,
excluded_other,309
excluded_public_support_proxy,10
final_test_only,3
shared_training,430
validation_maximization_only,6
validation_policy_only,2


Participant and question manifests saved before reading response values.


## 9. Partition the numeric responses without inspecting their distributions

We retain the original **one-based option positions**. Missing responses stay null; no imputation or feature fitting occurs. These are wide numeric partitions, not the later long-format, context-complete training dataset.

The helper below reads only requested columns, then selects exactly the fixed participant IDs. Checks test option validity and row identity without displaying any person's answers. The only exported earlier test answers are a separate final-panel repeatability reference, never persona input.


In [9]:
def partition_numeric(source_path, ids, columns):
    selected = pd.read_csv(source_path, usecols=["pid"] + columns, dtype="string")
    assert selected["pid"].is_unique
    selected = selected.set_index("pid").loc[ids, columns].reset_index()
    assert selected["pid"].tolist() == ids
    for column in columns:
        numeric = pd.to_numeric(selected[column], errors="raise")
        question = column_to_question[column]
        options = question.get("Options") if question["QuestionType"] == "MC" else question.get("Columns")
        assert options, f"No option metadata: {column}"
        observed = numeric.dropna()
        assert ((observed % 1) == 0).all(), f"Noninteger option position: {column}"
        assert observed.between(1, len(options)).all(), f"Invalid option position: {column}"
        selected[column] = numeric.astype("Int64")
    return selected

partition_specs = [
    ("shared_training", PROCESSED / "shared_training_wave1_3.parquet", wave13_path, train_ids, TRAINING_COLUMNS, "training"),
    ("development_anchors", PROCESSED / "development_anchors_wave1_3.parquet", wave13_path, development_ids, ANCHORS, "persona_evidence"),
    ("final_anchors", PROCESSED / "final_anchors_wave1_3.parquet", wave13_path, final_ids, ANCHORS, "persona_evidence"),
    ("validation_policy", HELDOUT / "validation_policy_wave4.parquet", wave4_path, development_ids, VALIDATION_POLICY, "validation_labels"),
    ("validation_maximization", HELDOUT / "validation_maximization_wave1_3.parquet", wave13_path, development_ids, VALIDATION_MAXIMIZATION, "validation_labels"),
    ("final_test", HELDOUT / "final_test_wave4.parquet", wave4_path, final_ids, FINAL_TEST, "final_test_labels"),
    ("final_retest_reference", HELDOUT / "final_test_original_wave1_3.parquet", wave13_path, final_ids, FINAL_TEST, "retest_reference_only"),
]
partition_summary = []
for name, destination, source, ids, columns, purpose in partition_specs:
    frame = partition_numeric(source, ids, columns)
    if purpose in {"training", "persona_evidence"}:
        assert not (set(frame.columns) - {"pid"}) & FORBIDDEN_INPUT_COLUMNS
        assert source == wave13_path
    save_fixed_parquet(destination, frame)
    partition_summary.append({
        "partition": name, "path": str(destination.relative_to(ROOT)), "purpose": purpose,
        "people": len(frame), "answer_columns": len(columns),
        "observed_answers": int(frame[columns].notna().sum().sum()),
        "missing_answers": int(frame[columns].isna().sum().sum()),
        "sha256": sha256_file(destination),
    })
    del frame

summary_frame = pd.DataFrame(partition_summary)
display(summary_frame[["partition", "people", "answer_columns", "observed_answers", "missing_answers"]])
print("No answer values, category counts, support shares or outcome-based selection were displayed or computed.")


,partition,people,answer_columns,observed_answers,missing_answers
0,shared_training,1658,430,712827,113
1,development_anchors,300,92,27576,24
2,final_anchors,100,92,9198,2
3,validation_policy,300,2,600,0
4,validation_maximization,300,6,1800,0
5,final_test,100,3,300,0
6,final_retest_reference,100,3,300,0


No answer values, category counts, support shares or outcome-based selection were displayed or computed.


## 10. Check saved partitions and record the audit

These checks verify separation, source integrity, reproducibility and missing-data preservation. They are structural checks, not model evaluation. The human test answers remain reserved for the final evaluation after predictions are frozen.


In [10]:
# Verify the saved data, rather than just the in-memory construction.
for name, destination, source, ids, columns, purpose in partition_specs:
    saved = pd.read_parquet(destination)
    assert saved["pid"].tolist() == ids
    assert saved.columns.tolist() == ["pid"] + columns
    assert saved["pid"].is_unique
    if purpose in {"training", "persona_evidence"}:
        assert not set(columns) & FORBIDDEN_INPUT_COLUMNS
    if purpose == "training":
        assert not set(saved["pid"]) & (set(development_ids) | set(final_ids))
    del saved

rerun_ids = np.random.default_rng(SEED).permutation(np.array(sorted(wave13_ids.tolist()), dtype=str))
assert rerun_ids.tolist() == train_ids + development_ids + final_ids
for entry in source_manifest["files"]:
    assert sha256_file(ROOT / entry["local_path"]) == entry["sha256"]

audit = {
    "status": "passed", "source_repo": REPO_ID, "source_revision": REVISION,
    "seed": SEED, "respondents": {name: len(ids) for name, ids in respondent_groups.items()},
    "training_question_count": len(TRAINING_COLUMNS), "persona_evidence_question_count": len(ANCHORS),
    "validation_question_count": len(VALIDATION_POLICY) + len(VALIDATION_MAXIMIZATION),
    "test_question_count": len(FINAL_TEST),
    "partitions": partition_summary,
    "checks": {
        "source_checksums_match": True, "respondent_groups_disjoint": True,
        "respondent_split_reproducible": True, "unique_answer_column_mapping": True,
        "anchors_subset_of_training_questions": True,
        "heldout_families_absent_from_training_and_anchors": True,
        "no_wave4_input_answers": True, "saved_option_positions_valid": True,
        "no_outcome_based_split_selection": True,
    },
    "test_labels_handling": "Read programmatically only to partition and check option validity; never displayed, summarized by category, or used for fitting or selection.",
    "limitations": [
        "Raw downloads contain held-out labels; downstream training must load the explicit processed partitions only.",
        "Semantic-equivalence auditing and complete task-context reconstruction are still required before feature generation.",
        "This is a historical Prolific cohort; equal panel weights are not national survey weights.",
    ],
    "training_started": False,
}
save_fixed_json(SPLITS / "partition_manifest.json", audit)
save_fixed_json(REPORTS / "data_split_audit.json", audit)
print("PASS: source integrity, saved-file schemas, disjoint people, allowed questions, and reproducible splitting.")


PASS: source integrity, saved-file schemas, disjoint people, allowed questions, and reproducible splitting.


## Result and next step

- `data/raw/twin_2k_500/`: original pinned source files, documentation and checksums.
- `data/splits/`: respondent assignments, question roles and partition audit.
- `configs/data_source.json` and `configs/data_split_plan.json`: source revision and reproducible configuration, suitable for version control.
- `data/processed/`: shared-training responses and permitted development/final persona evidence.
- `data/heldout/`: validation labels, final-test labels and the separate earlier-answer repeatability reference.
- `outputs/data_split_audit.json`: structural verification results.

**Next:** build the context-complete question bank and cleaned long-format training records from these partitions. Preserve introductory instructions for monetary choices and games, map one-based options carefully, and perform the remaining semantic leakage audit. Do not use all raw response columns as model input. No model has been trained or scored in this notebook.
